In [12]:
!pip install plotly ipywidgets

In [13]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

In [14]:
df = pd.read_csv(r"C:\Users\DELL\Downloads\Cleaned_Data.csv")

df.columns = df.columns.str.strip()
df["Date"] = pd.to_datetime(df["Date"])

df.head()

,Region,Date,Frequency,Estimated Unemployment Rate (%),Estimated Employed,Estimated Labour Participation Rate (%),Area,Year,Month,Month_Name,Covid_Period
0,Andhra Pradesh,2019-05-31,Monthly,3.65,11999139.0,43.24,Rural,2019,5,May,Pre-Covid
1,Andhra Pradesh,2019-06-30,Monthly,3.05,11755881.0,42.05,Rural,2019,6,June,Pre-Covid
2,Andhra Pradesh,2019-07-31,Monthly,3.75,12086707.0,43.50,Rural,2019,7,July,Pre-Covid
3,Andhra Pradesh,2019-08-31,Monthly,3.32,12285693.0,43.97,Rural,2019,8,August,Pre-Covid
4,Andhra Pradesh,2019-09-30,Monthly,5.17,12256762.0,44.68,Rural,2019,9,September,Pre-Covid


In [15]:
region_dropdown = widgets.Dropdown(
    options=["All"] + sorted(df["Region"].unique().tolist()),
    value="All",
    description="Region:"
)

area_dropdown = widgets.Dropdown(
    options=["All"] + sorted(df["Area"].unique().tolist()),
    value="All",
    description="Area:"
)

output = widgets.Output()

In [16]:
def update_dashboard(region, area):
    with output:
        clear_output(wait=True)

        filtered_df = df.copy()

        if region != "All":
            filtered_df = filtered_df[filtered_df["Region"] == region]

        if area != "All":
            filtered_df = filtered_df[filtered_df["Area"] == area]

        avg_unemployment = filtered_df["Estimated Unemployment Rate (%)"].mean()
        avg_labour = filtered_df["Estimated Labour Participation Rate (%)"].mean()
        avg_employed = filtered_df["Estimated Employed"].mean()

        print("UNEMPLOYMENT DASHBOARD - INDIA")
        print("--------------------------------")
        print(f"Average Unemployment Rate: {avg_unemployment:.2f}%")
        print(f"Average Labour Participation Rate: {avg_labour:.2f}%")
        print(f"Average Estimated Employed: {avg_employed:,.0f}")

        monthly = (
            filtered_df.groupby("Date")["Estimated Unemployment Rate (%)"]
            .mean()
            .reset_index()
        )

        fig1 = px.line(
            monthly,
            x="Date",
            y="Estimated Unemployment Rate (%)",
            markers=True,
            title="Average Unemployment Rate Over Time"
        )
        fig1.show()

        region_avg = (
            filtered_df.groupby("Region")["Estimated Unemployment Rate (%)"]
            .mean()
            .sort_values(ascending=False)
            .reset_index()
        )

        fig2 = px.bar(
            region_avg,
            x="Estimated Unemployment Rate (%)",
            y="Region",
            orientation="h",
            title="Average Unemployment Rate by Region"
        )
        fig2.show()

        area_trend = (
            filtered_df.groupby(["Date", "Area"])["Estimated Unemployment Rate (%)"]
            .mean()
            .reset_index()
        )

        fig3 = px.line(
            area_trend,
            x="Date",
            y="Estimated Unemployment Rate (%)",
            color="Area",
            markers=True,
            title="Rural vs Urban Unemployment Trend"
        )
        fig3.show()

        def covid_period(date):
            if date < pd.to_datetime("2020-03-01"):
                return "Pre-Covid"
            elif date <= pd.to_datetime("2020-06-30"):
                return "Covid Period"
            else:
                return "Post-Covid"

        filtered_df["Covid_Period"] = filtered_df["Date"].apply(covid_period)

        covid_avg = (
            filtered_df.groupby("Covid_Period")["Estimated Unemployment Rate (%)"]
            .mean()
            .reset_index()
        )

        fig4 = px.bar(
            covid_avg,
            x="Covid_Period",
            y="Estimated Unemployment Rate (%)",
            color="Covid_Period",
            title="Covid-19 Impact on Unemployment"
        )
        fig4.show()

        display(filtered_df.head(10))

In [17]:
def on_filter_change(change):
    update_dashboard(region_dropdown.value, area_dropdown.value)

region_dropdown.observe(on_filter_change, names="value")
area_dropdown.observe(on_filter_change, names="value")

In [18]:
display(region_dropdown, area_dropdown, output)

update_dashboard(region_dropdown.value, area_dropdown.value)

Dropdown(description='Region:', options=('All', 'Andhra Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgar…

Dropdown(description='Area:', options=('All', 'Rural', 'Urban'), value='All')

Output()